# The coarse-correction step: what a finer level actually receives

The per-level panels (wandb `val/levels`, `vcycle_levels_real.ipynb`) show each level's
recon $w_\ell$. A finer level never uses that image. On the way up, `PDVCycle` adds

$$x_\ell \;\leftarrow\; x_\ell + \alpha_x\, P\big(w_{\ell+1} - u_{\ell+1}\big), \qquad u_{\ell+1} = R\,x_\ell ,$$

so only the coarse **change** crosses, and only **through $P$**. A coarse recon can be
badly wrong and still hand up a harmless correction if:
* its error sits in the band $P$ does not pass (the coarse grid's top band; $|H| = 1/2$
  at the coarse Nyquist and falls to 0 at the fine one), or
* the error was already in its starting point $u_{\ell+1}$, so it cancels in $w - u$, or
* $\alpha_x$ is small.

In each of those cases, nothing in the loss pushes the coarse level back to an image.

Levels are numbered as in the code and the wandb panel: **L0 = finest**, L2 = coarsest.
Each row below is one transition $L_{\ell+1} \to L_\ell$, coarsest first.

Engine: `scripts/level_figure.py` (`correction_parts`, `correction_table`,
`fig_correction`, `fig_correction_spectra`). The measurement is `eval_per_volume`'s for
the slice. Live mode only: it needs the checkpoint, the data, and torch for $P$.

In [ ]:
import pathlib, sys, os

_root = pathlib.Path.cwd()
if not (_root / "models").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "scripts"))
os.chdir(_root)                      # run dirs and the data roots are repo-relative

import numpy as np
import matplotlib.pyplot as plt
import importlib
import level_figure as lf
if getattr(lf, "_FILE_MTIME", None) != os.path.getmtime(lf.__file__):
    lf = importlib.reload(lf)
    lf._FILE_MTIME = os.path.getmtime(lf.__file__)

RUN     = "trained_nets/mg_recon/knee/mg81v6vtxA_R8"   # still has the missing-pi^1 FAS
VOLUME  = "1000108"         # any unique part of a knee val volume name
SLICE   = 22
SIGMA   = None              # None: the run's val_noise_std
VAL_KSPACE = "measurement_awgn"
OUTER   = -1                # which V-cycle (0-based; -1 = the last)
FLIP    = None              # None: 'ud' for knee, as the viewer

In [ ]:
z, meta, live = lf.trace_slice(RUN, VOLUME, SLICE, sigma=SIGMA, val_kspace_type=VAL_KSPACE)
FLIP = lf.default_flip(meta) if FLIP is None else FLIP
print(f"{meta['run']}  step {meta['step']}  {meta['n_params']:,} params   "
      f"{meta['volume']} slice {meta['slice']}  R={meta['R']}  sigma={meta['sigma']:g}")
print(f"transfer filter: {meta['transfer_filter']}   coarse op: {meta['coarse_op']}   "
      f"output NRMSE {100 * lf.nrmse(z['recon'], z['gt']):.2f}%")

## 1. The correction step, image domain

Columns:
1. $w_c$, the coarse recon.
2. $R\,\mathrm{gt}$ on the coarse grid.
3. The coarse error $|w_c - R\,\mathrm{gt}|$.
4. $P(w_c)$.
5. The part of the coarse error that $P$ lets through, $|P(w_c - R\,\mathrm{gt})|$.
6. What the coarse level hands up, $|P(w_c - u_c)|$.
7. What the fine level actually adds, $|\alpha_x P(\cdot)|$.
8. The fine level before the correction.
9. The fine level after it.

How the panels are scaled:
* Columns 3 and 5 share the error window (vmax / 5), so they compare directly.
* Columns 6 and 7 share their own window, printed in cyan, because a correction is much
  smaller than an error. Cyan also gives each one's RMS as a % of the fine image.
* Yellow is NRMSE. $P(w_c)$ is scored against $P(R\,\mathrm{gt})$, which isolates the coarse
  error that survives $P$.

The row label gives two numbers:
* **visible**: RMS of $P(\text{coarse error})$ / RMS of the coarse error. $P$ keeps the RMS of
  anything in its passband, so 100% means the whole error crosses.
* **alpha**: fitted from added = $\alpha \cdot P(w_c - u_c)$. The fit's misfit is in the table
  below; it is ~0 when the formula above holds.

In [ ]:
fig = lf.fig_correction(z, meta, outer=OUTER, flip=FLIP)
plt.show()

## 2. The same numbers for every V-cycle

* `nrmse_w_c`: the coarse recon against $R\,\mathrm{gt}$, which is what the panels show.
* `nrmse_P_w`: $P(w_c)$ against $P(R\,\mathrm{gt})$.
* `visible`: as above.
* `step_rel`: RMS of what the fine level adds / RMS of the fine image.
* `alpha` and `fit misfit`: $\alpha_x$ recovered from the trace.

If the coarse recon degrades over V-cycles but `visible` × `nrmse_w_c` and `step_rel` stay
small, the fine levels are not being fed the corruption.

In [ ]:
rows = lf.correction_table(z, meta)
print(f"{'V':>2} {'transition':>10} {'nrmse w_c':>10} {'nrmse P(w_c)':>13} {'visible':>8} "
      f"{'step_rel':>9} {'alpha':>8} {'fit misfit':>10}")
for r in rows:
    print(f"{r['outer']:>2} {r['transition']:>10} {100 * r['nrmse_w_c']:>9.2f}% "
          f"{100 * r['nrmse_P_w']:>12.2f}% {100 * r['visible']:>7.1f}% "
          f"{100 * r['step_rel']:>8.3f}% {r['alpha'].real:>+8.3f} {r['alpha_misfit']:>10.1e}")

## 3. Frequency content: the coarse recon vs what crosses $P$

Each row is one transition:
* **Four 2-D spectra**, $20\log_{10}|\mathcal{F}|$: of $w_c$, of $P(w_c)$, of the coarse error,
  and of $P$(coarse error). All four are on one axis in cycles per **fine** pixel, so the
  coarse grid's whole band is the dashed box.
* **1-D power along each axis**: the fine $R^\ell\mathrm{gt}$, $w_c$, $P(w_c)$, and the two
  errors. These are summed over the other axis, so an image and its interpolation agree
  inside the band.
* **Bottom row**: $|H|$ of $P$, which explains the gap between $w_c$ and $P(w_c)$ near
  the coarse Nyquist.

How to read it:
* A coarse error concentrated near the edge of the box, or along the coarse Nyquist
  lines, is the part $P$ mostly removes.
* A coarse error near the centre crosses at full strength.

In [ ]:
fig = lf.fig_correction_spectra(z, meta, outer=OUTER, flip=FLIP)
plt.show()

## 4. Every V-cycle

The correction figure for each outer V-cycle, to see when the coarse recon starts to drift
and whether what crosses drifts with it.

In [ ]:
SHOW_ALL = False
if SHOW_ALL:
    for k in range(len(meta["outers"])):
        fig = lf.fig_correction(z, meta, outer=k, flip=FLIP, panel=1.6)
        plt.show()